In [ ]:
# using 'venv_train_llama' venv from ~/projects/ai-ml-exploration/notebooks/model_training_genai 

In [3]:
!pip install scikit-learn

  Using cached scipy-1.16.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (61 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 11.2 MB/s eta 0:00:00a 0:00:01
Using cached joblib-1.5.2-py3-none-any.whl (308 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 13.1 MB/s eta 0:00:0000:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
import shutil
from PIL import Image, ImageFile
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Allow loading truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Paths
CLEAN_DATA_DIR = "~/dev/oral_cavity_dataset_clean"
RUNS_DIR = './runs'



## Load Datasets

In [5]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Transforms
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Then load datasets from the cleaned folders
train_dataset = datasets.ImageFolder(root=os.path.join(CLEAN_DATA_DIR, 'train'), transform=train_transforms)
val_dataset = datasets.ImageFolder(root=os.path.join(CLEAN_DATA_DIR, 'val'), transform=val_test_transforms)
test_dataset = datasets.ImageFolder(root=os.path.join(CLEAN_DATA_DIR, 'test'), transform=val_test_transforms)

# DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)


In [6]:
# Class names
print(train_dataset.classes)  # Should print ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown']

['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown']


## Train Model

You Check Tensorborad logs with following command

`tensorboard --logdir=./runs`

Then open browser: http://localhost:6006

In [7]:

# Create Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = models.mobilenet_v3_large(pretrained=True)
num_classes = 5  # Benign, OPMD, OCA, Healthy, Unknown
model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# TensorBoard
writer = SummaryWriter(log_dir=RUNS_DIR)

# Early stopping parameters
patience = 5
best_val_loss = float('inf')
early_stopping_counter = 0

# Training
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = correct / total
    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_acc = correct / total
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

    writer.add_scalar('Loss/Train', avg_train_loss, epoch)
    writer.add_scalar('Loss/Val', avg_val_loss, epoch)
    writer.add_scalar('Accuracy/Train', train_acc, epoch)
    writer.add_scalar('Accuracy/Val', val_acc, epoch)

    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')  # Save best model
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

writer.close()


Using device: cpu
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /Users/alokdwivedi/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


/Users/alokdwivedi/dev/github/medpylabs/oral_cavity_classifier_app/ml/training/venv_oral_cavity_pytorch/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/alokdwivedi/dev/github/medpylabs/oral_cavity_classifier_app/ml/training/venv_oral_cavity_pytorch/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100.0%
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x10bcf6b60>
Traceback (most recent call last):
  File "/Users/alokdwivedi/dev/github/medp

KeyboardInterrupt: 

## Save Model

In [12]:
model_path = "./dataset/oral_cavity_dataset/mobilenetv3_oral_cavity_large2.pth"
torch.save(model.state_dict(), model_path)
print("Mobilenetv3 Oral Cavity Large Model saved!")

Mobilenetv3 Oral Cavity Large Model saved!


## Load and test model accuracy

In [27]:
# Re-create model
model = models.mobilenet_v3_large(pretrained=False)
num_classes = 5
model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

# Load the saved weights
model.load_state_dict(torch.load(model_path))

# Move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

/tmp/ipykernel_3991124/3760665820.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [14]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:  # test_loader you already created
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

test_accuracy = correct / total
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

Test Accuracy: 72.47%


## Image label prediction

In [2]:
import torch
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import sys

# Define your class labels (update as per your model's training)
class_labels = ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown']

# Load your model
def load_model(model_path):
    model = models.mobilenet_v3_large(pretrained=False)  # or mobilenet_v3_small if you used small
    model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, len(class_labels))
    model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
    model.eval()
    return model

# Preprocess image
def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),  # MobileNetV3 expects 224x224
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    image = Image.open(image_path).convert('RGB')
    image = transform(image)
    image = image.unsqueeze(0)  # Add batch dimension
    return image

# Predict
def predict(model, image_tensor):
    with torch.no_grad():
        outputs = model(image_tensor)
        _, predicted = torch.max(outputs, 1)
        return class_labels[predicted.item()]



In [4]:
model_path = "./dataset/oral_cavity_dataset/mobilenetv3_oral_cavity_large2.pth"

In [5]:
image_base_path = "dataset/oral_cavity_dataset/cnn/prepared_dataset/test"
label = "Healthy"
image_name = "C-4-6-2.jpg"

image_path = os.path.join(image_base_path,label,image_name)

model = load_model(model_path)
image_tensor = preprocess_image(image_path)
prediction = predict(model, image_tensor)

print(f"Predicted label: {prediction}")
print(f"Expected label: {label}")

/home/ant/projects/ai-ml-exploration/notebooks/model_training_genai/venv_train_llama/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ant/projects/ai-ml-exploration/notebooks/model_training_genai/venv_train_llama/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/tmp/ipykernel_3230162/3388693096.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECU

Predicted label: Benign
Expected label: Healthy


In [6]:
image_base_path = "raw_kaggle_downloads/oral_cancer_dataset03"
label = "cancer"
image_name = "mouth-cancer.jpg"

image_path = os.path.join(image_base_path,label,image_name)

model = load_model(model_path)
image_tensor = preprocess_image(image_path)
prediction = predict(model, image_tensor)

print(f"Predicted label: {prediction}")
print(f"Expected label: {label}")

Predicted label: Benign
Expected label: cancer


/tmp/ipykernel_3230162/3388693096.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))


In [7]:
image_base_path = "raw_kaggle_downloads/oral_cancer_dataset03"
label = "non-cancer"
image_name = "tongue.jpg"
#image_name = "tongue-tingling-in-person-with-their-tongue-sticking-out.jpg"

image_path = os.path.join(image_base_path,label,image_name)

model = load_model(model_path)
image_tensor = preprocess_image(image_path)
prediction = predict(model, image_tensor)

print(f"Predicted label: {prediction}")
print(f"Expected label: {label}")

Predicted label: Benign
Expected label: non-cancer


/tmp/ipykernel_3230162/3388693096.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))


In [8]:
image_base_path = "dataset/oral_cavity_unknown"
label = "Unknown"
image_name = "000000581781.jpg"
image_path = os.path.join(image_base_path,image_name)

model = load_model(model_path)
image_tensor = preprocess_image(image_path)
prediction = predict(model, image_tensor)

print(f"Predicted label: {prediction}")
print(f"Expected label: {label}")

Predicted label: Unknown
Expected label: Unknown


/tmp/ipykernel_3230162/3388693096.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))


## Convert PyTorch Model to ONNX 

In [34]:
import torch
from torchvision import models
import torch.nn as nn

# Define and load model
model = models.mobilenet_v3_large(pretrained=False)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 5)
model.load_state_dict(torch.load(model_path, map_location='cpu'))
model.eval()

# Dummy input: batch_size=1, channels=3, height=224, width=224
dummy_input = torch.randn(1, 3, 224, 224)

# Export to ONNX format
onnx_model_path = './dataset/oral_cavity_dataset/mobilenetv3_oral_cavity.onnx'
torch.onnx.export(
    model,
    dummy_input,
    onnx_model_path,
    input_names=["input"],
    output_names=["output"],
    opset_version=11,
    do_constant_folding=True
)

/tmp/ipykernel_3991124/2153875228.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location='cpu'))


## Convert ONNX Model to Tensorflow Lite

Run this in a separate venv
```shell
python3 -m venv venv_onnx_convert_tflite
source venv_onnx_convert_tflite/bin/activate

pip install \
  tensorflow==2.13.0 \
  keras==2.13.1 \
  tensorflow-addons==0.23.0 \
  "tensorflow-probability[tf]==0.21.0" \
  onnx==1.17.0 \
  onnx-tf==1.10.0
```

In [1]:
import tensorflow as tf
import keras
import onnx
from onnx_tf.backend import prepare
import tensorflow_addons as tfa
import tensorflow_probability as tfp

print("✅ All modules imported successfully.")

2025-05-03 13:53:08.283645: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-03 13:53:08.285425: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-03 13:53:08.328719: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-03 13:53:08.329912: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-03 13:53:09.036033: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

✅ All modules imported successfully.


/home/ant/projects/ai-ml-exploration/notebooks/model_training_genai/venv_onnx_convert_tflite/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [5]:
import onnx
from onnx_tf.backend import prepare
import tensorflow as tf

# Load ONNX model
onnx_model_path = './dataset/oral_cavity_dataset/mobilenetv3_oral_cavity.onnx'
onnx_model = onnx.load(onnx_model_path)

# Convert ONNX to TensorFlow
tf_rep = prepare(onnx_model)
tf_model_folder = './dataset/oral_cavity_dataset/mobilenetv3_oral_cavity_tf_model'
tf_rep.export_graph(tf_model_folder)

# Convert TensorFlow model to TFLite
converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_folder)
tflite_model = converter.convert()

# Save the TFLite model
tflite_model_path = './dataset/oral_cavity_dataset/mobilenetv3_oral_cavity_model.tflite'
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: ./dataset/oral_cavity_dataset/mobilenetv3_oral_cavity_tf_model/assets


INFO:tensorflow:Assets written to: ./dataset/oral_cavity_dataset/mobilenetv3_oral_cavity_tf_model/assets
2025-05-05 20:25:20.648928: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2025-05-05 20:25:20.648956: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2025-05-05 20:25:20.649513: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: ./dataset/oral_cavity_dataset/mobilenetv3_oral_cavity_tf_model
2025-05-05 20:25:20.661221: I tensorflow/cc/saved_model/reader.cc:91] Reading meta graph with tags { serve }
2025-05-05 20:25:20.661261: I tensorflow/cc/saved_model/reader.cc:132] Reading SavedModel debug info (if present) from: ./dataset/oral_cavity_dataset/mobilenetv3_oral_cavity_tf_model
2025-05-05 20:25:20.681698: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:375] MLIR V1 optimization pass is not enabled
2025-05-05 20:25:20.683856: I tensorflow/cc/saved_mode

##  Test Converted TFLite model

In [1]:
tflite_model_path = './dataset/oral_cavity_dataset/mobilenetv3_oral_cavity_model.tflite'

In [12]:
import numpy as np
import tensorflow as tf
from PIL import Image
#from torchvision import transforms # Only if you want to reuse your PyTorch transforms directly

# Define your class labels (ensure they match your model's training)
class_labels = ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown']


# Load the TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Model Input Details:", input_details)
print("TFLite Model Output Details:", output_details)

def preprocess_image_for_tflite(image_path, input_details):
    # Get expected input shape from TFLite model details
    input_shape = input_details[0]['shape'] # This is [1, 3, 224, 224] (NCHW)
    input_dtype = input_details[0]['dtype'] # numpy.float32

    img = Image.open(image_path).convert('RGB') # Ensure RGB

    # Resize the image to the model's expected dimensions (H, W)
    # For NCHW, H=input_shape[2], W=input_shape[3]
    img = img.resize((input_shape[3], input_shape[2])) # Resize to (width, height) = (224, 224)
    img_np = np.array(img, dtype=input_dtype) # This will be (H, W, C) = (224, 224, 3)

    # Normalize pixel values to [0.0, 1.0]
    img_np = img_np / 255.0

    # Add batch dimension (from [224, 224, 3] to [1, 224, 224, 3])
    input_tensor = np.expand_dims(img_np, axis=0)

    # CRITICAL: Transpose from NHWC ([1, H, W, C]) to NCHW ([1, C, H, W])
    # The model expects [1, 3, 224, 224]
    input_tensor = np.transpose(input_tensor, (0, 3, 1, 2)) # (Batch, Channels, Height, Width)

    # --- Optional: Apply mean and std normalization IF the TFLite model expects it externally. ---
    # Based on your PyTorch training, this was done externally.
    # If your TFLite model was converted to include normalization in its graph,
    # you *might* not need to do it here. However, for float32 models converted
    # from PyTorch/Keras, this is often still an external preprocessing step.
    # Uncomment if needed after testing without it.
    mean = np.array([0.485, 0.456, 0.406], dtype=input_dtype).reshape((1, 3, 1, 1))
    std = np.array([0.229, 0.224, 0.225], dtype=input_dtype).reshape((1, 3, 1, 1))
    input_tensor = (input_tensor - mean) / std

    return input_tensor


TFLite Model Input Details: [{'name': 'serving_default_input:0', 'index': 0, 'shape': array([  1,   3, 224, 224], dtype=int32), 'shape_signature': array([  1,   3, 224, 224], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
TFLite Model Output Details: [{'name': 'PartitionedCall:0', 'index': 450, 'shape': array([1, 5], dtype=int32), 'shape_signature': array([1, 5], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [14]:
# TFLite Model test with one image 

import os

image_base_path = "dataset/oral_cavity_dataset/cnn/prepared_dataset/test"
actual_label = "Healthy"
image_name = "C-4-6-2.jpg"
test_image_path = os.path.join(image_base_path,actual_label,image_name)

input_data = preprocess_image_for_tflite(test_image_path, input_details)

# Set the input tensor
interpreter.set_tensor(input_details[0]['index'], input_data)

# Run inference
interpreter.invoke()

# Get the output tensor
output_data = interpreter.get_tensor(output_details[0]['index'])

# ************** CRITICAL CHANGE HERE **************
# Apply Softmax to convert logits to probabilities
# output_data[0] contains the logits for the first (and only) item in the batch
probabilities = tf.nn.softmax(output_data[0]).numpy()

predicted_index = np.argmax(probabilities)
predicted_label = class_labels[predicted_index]
confidence = probabilities[predicted_index] * 100 # Now, probabilities are between 0 and 1

print(f"Expected label: {actual_label}")
print(f"Predicted Label (TFLite Python): {predicted_label}")
print(f"Confidence (TFLite Python): {confidence:.2f}%")


Expected label: Healthy
Predicted Label (TFLite Python): Benign
Confidence (TFLite Python): 94.20%


2025-05-11 19:55:14.062561: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Embedding TFLIte in Mobile App

To complete the mobile-side inference code using TensorFlow.js in a React Native app, we need to:
	1.	Preprocess the image similarly to your Python code.
	2.	Load the .tflite model via react-native-tflite or a supported mobile inference engine.
	3.	Normalize and resize the image to match your training input (224x224, RGB, normalized).

Since TensorFlow.js in React Native does not support .tflite models directly, you should use react-native-tflite for local inference with .tflite models. Here’s a complete solution using it:

⸻

✅ Step-by-Step: React Native Inference with TFLite

1. Install the required package

npm install react-native-tflite
npx pod-install

2. Link the native module (RN < 0.60 only)

react-native link react-native-tflite



⸻

3. Code for Loading and Using the Model

import Tflite from 'react-native-tflite';
import { Image, View } from 'react-native';
import { launchImageLibrary } from 'react-native-image-picker';
import { decodeJpeg, resizeBilinear, cast, expandDims, tensor } from '@tensorflow/tfjs-react-native';

const tflite = new Tflite();

// Load model on startup
const loadModel = () => {
  tflite.loadModel(
    {
      model: 'mobilenetv3_oral_cavity_model.tflite',
      labels: 'labels.txt',
    },
    (err, res) => {
      if (err) console.log('Error loading model', err);
      else console.log('Model loaded');
    }
  );
};



⸻

4. Preprocess Image and Run Inference

const runInference = (imagePath) => {
  tflite.runModelOnImage(
    {
      path: imagePath,
      imageMean: 127.5,    // Matches (0.485 * 255) + (0.229 * 255) / 2
      imageStd: 127.5,     // Roughly matches torchvision standard normalization
      numResults: 5,
      threshold: 0.1,
    },
    (err, results) => {
      if (err) console.log('Inference error', err);
      else console.log('Predictions:', results);
    }
  );
};



⸻

🏷️ labels.txt file (must match model training labels)

Benign
Healthy
OCA
OPMD
Unknown

Place this file in your project’s assets or android/app/src/main/assets depending on the platform.

⸻

📷 Image Input (from camera/gallery)

launchImageLibrary({}, (response) => {
  if (response.assets && response.assets.length > 0) {
    const imagePath = response.assets[0].uri.replace('file://', '');
    runInference(imagePath);
  }
});



⸻

📌 Notes
	•	imageMean and imageStd are for normalizing pixel values. If you’re using standard torchvision normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), the TFLite quantization must match.
	•	If you used quantized TFLite model, change inference method to runModelOnImage with quantized=true.

⸻

Would you like help generating the .tflite model from your PyTorch .pth file using ONNX → TensorFlow → TFLite workflow?